# SMB-lending diligence: runway and ranking

This notebook recomputes cash runway for every company in `analysis/comparison.json` and ranks the cohort for the Q3 2026 credit committee.

**Runway (months) = cash and equivalents / average monthly net burn**, both in USD millions and both cited in the table.

In [ ]:
import json
from pathlib import Path

table = json.loads(Path("comparison.json").read_text())
rows = table["rows"]
print(len(rows), "companies as of", table["asOf"])

## Recompute runway

The stored `runwayMonths` must equal cash / burn within rounding (0.1 month). Any mismatch means the table was edited by hand.

In [ ]:
for r in rows:
    runway = r["cashUsdM"]["value"] / r["monthlyBurnUsdM"]["value"]
    assert abs(runway - r["runwayMonths"]["value"]) <= 0.1, r["company"]
    r["runway_check"] = round(runway, 1)
[(r["company"], r["runway_check"]) for r in rows]

## Ranking

Candidates are ranked by runway, longest first (committee criteria). Runway below 12 months needs an equity commitment or a liquidity covenant; below 9 months is a decline unless new equity is committed.

In [ ]:
def band(runway):
    return "proceed" if runway >= 12 else ("conditions" if runway >= 9 else "decline unless new equity")

ranked = sorted(rows, key=lambda r: (-r["runway_check"], r["company"]))
for i, r in enumerate(ranked, 1):
    print(f"{i:>2}. {r['company']:<22} {r['runway_check']:>5.1f} months  {band(r['runway_check'])}")

## Exceptions the ranking does not show

- Tessaly Credit: ARR restated to $11.6M after removing one-time pilot revenue; growth restated to 40%.
- Ynderra Lending: largest customer is 46% of ARR, above the 25% concentration guideline.
- Marrowfield Credit: figures come from the v2 statement; the v1 draft is superseded.
- Restricted documents (board minutes, litigation memo) were unavailable and are not reflected in runway.

In [ ]:
flags = {r["company"]: r["topRisk"]["text"] for r in rows if any(k in r["topRisk"]["text"].lower() for k in ("restat", "concentration"))}
flags